# Layer 26 SAE 语料审计
只读检查源目录；token 是固定种子 reservoir 估计。完整方法和限制见 REPORT.md。
以下代码嵌入实际执行的脚本，末尾展示已保存结果。重跑会覆盖本审计目录的统计文件，不改动原始语料。

## audit.py

```python
"""Read-only corpus census; token totals are seeded reservoir estimates, not exact counts."""
import csv, hashlib, json, os, random, re, statistics
from collections import Counter
from pathlib import Path
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
from tokenizers import Tokenizer

ROOT = Path('/home/haoqian/Data/past/Molecule')
OUT = Path(__file__).resolve().parent
TOK = Tokenizer.from_file('/mnt_nas1/haoqian/Data/Molecule/chemical_models/ChemDFM-R-14B/tokenizer.json')
N_SAMPLE = 1024

def array_rows(path):
    decoder = json.JSONDecoder()
    with open(path) as f:
        buf, pos, eof = '', 0, False
        while True:
            if len(buf) - pos < 65536 and not eof:
                chunk = f.read(1048576)
                buf, pos, eof = buf[pos:] + chunk, 0, not chunk
            while pos < len(buf) and buf[pos] in ' \t\r\n[,':
                pos += 1
            if pos == len(buf):
                if eof: return
                continue
            if buf[pos] == ']': return
            try:
                row, pos = decoder.raw_decode(buf, pos)
            except json.JSONDecodeError:
                if eof: raise
                chunk = f.read(1048576)
                buf, pos, eof = buf[pos:] + chunk, 0, not chunk
                continue
            yield row

def rows(path):
    if path.suffix == '.json': yield from array_rows(path)
    elif path.suffix == '.jsonl':
        with open(path) as f:
            for line in f:
                if line.strip(): yield json.loads(line)
    else:
        with open(path) as f: yield from csv.DictReader(f, delimiter='\t' if path.suffix == '.txt' else ',')

def texts(r, kind):
    if kind == 'chemcot': return str(r.get('query') or ''), str(r.get('raw_cot') or '')
    if kind in ('mollama', 'moleculeqa'):
        qa = r.get('conversations', [])
        # Preserve dialog order; replace unavailable multimodal input with explicit SMILES.
        smi = str(r.get('smiles') or '')
        full = '\n'.join(str(t.get(k) or '').replace('<mol>', 'Molecule SMILES: '+smi) for t in qa for k in ('user','assistant'))
        ans = '\n'.join(str(t.get('assistant') or '') for t in qa)
        return full, ans
    if kind == 'pubchem': return str(r.get('description') or ''), str(r.get('enriched_description') or '')
    if kind == 'openmol': return str(r.get('Instruction') or ''), str(r.get('molecule') or '')
    if kind == 'chebi': return str(r.get('SMILES') or ''), str(r.get('description') or '')
    if kind == 'processed':
        inp, target = r.get('input') or {}, r.get('targets') or {}
        return str(inp.get('instruction') or ''), str(target.get('text') or '')
    raise ValueError(kind)

def estimate(lengths, n):
    mean = statistics.mean(lengths)
    se = statistics.stdev(lengths) / len(lengths)**.5 * max(0,1-len(lengths)/n)**.5 if len(lengths)>1 else 0
    ordered = sorted(lengths)
    return {'mean':mean, 'estimated_total':round(mean*n), 'approx_95pct_margin':round(1.96*se*n), 'sample_p50':ordered[len(ordered)//2], 'sample_p95':ordered[min(len(ordered)-1,int(.95*len(ordered)))]}

def audit(path,kind):
    rng, sample = random.Random(20260921), []
    seen, cids, tasks = set(), set(), Counter()
    chars = [0,0]; empty = [0,0]; markers=0; placeholders=0; turns=0
    for n,r in enumerate(rows(path),1):
        a,b = texts(r,kind)
        digest=hashlib.sha256((a+'\x00'+b).encode()).digest()[:16]
        seen.add(digest)
        for i,t in enumerate((a,b)):
            chars[i]+=len(t); empty[i]+=not bool(t.strip())
        if r.get('cid') is not None: cids.add(str(r['cid']))
        if kind in ('mollama','moleculeqa'):
            turns += len(r.get('conversations',[]))
            placeholders += any('<mol>' in str(t.get('user','')) for t in r.get('conversations',[]))
        tasks[str(r.get('SubTask') or r.get('subtask') or r.get('task_type') or r.get('category') or kind)] += 1
        if kind=='chemcot': markers += bool(re.search(r'ground[ -]truth|reference answer|correct answer is', b, re.I))
        if len(sample)<N_SAMPLE: sample.append((a,b))
        else:
            j=rng.randrange(n)
            if j<N_SAMPLE: sample[j]=(a,b)
    metrics={}
    for name, seq in [('first',[a for a,b in sample]),('second',[b for a,b in sample]),('combined',[(a+'\n'+b) if kind not in ('mollama','moleculeqa') else a for a,b in sample])]:
        lens=[len(x.ids) for x in TOK.encode_batch(seq,add_special_tokens=False)]
        metrics[name]=estimate(lens,n)
    result={'path':str(path.relative_to(ROOT)), 'kind':kind, 'bytes':path.stat().st_size,'rows':n,'unique_text_pairs':len(seen),'unique_cids':len(cids),'turns':turns,'empty_fields':empty,'text_characters':chars,'tasks':dict(tasks),'answer_cue_rows':markers,'mol_placeholder_rows':placeholders,'sample_n':len(sample),'tokens':metrics}
    print(json.dumps({k:result[k] for k in ('path','rows','unique_text_pairs','answer_cue_rows')})+' tokens='+str(metrics['combined']['estimated_total']),flush=True)
    return result,cids,seen

def main():
    jobs=[]
    jobs += [(p,'chemcot') for p in sorted((ROOT/'Downloads/Train/Chemcot').glob('*/*.json'))]
    base=ROOT/'Molecule/Baselines/Mol-LLaMA/data'
    jobs += [(base/'Mol-LLaMA-Instruct'/name,'mollama') for name in ['detailed_structural_descriptions.json','structure2chemical_features_relationships.json','structure2biological_features_relationships.json','comprehensive_conversations.json']]
    jobs += [(base/'Mol-LLaMA-Instruct/pubchem-molecules.json','pubchem')]
    jobs += [(ROOT/'Downloads/Train/OpenMolIns'/size/'train.csv','openmol') for size in ['xlarge','large','medium','small','light']]
    jobs += [(base/'moleculeqa/train.json','moleculeqa'),(base/'ChEBI-20/train.txt','chebi')]
    jobs += [(p,'processed') for p in sorted((ROOT/'Datasets/Train').glob('*/*.jsonl'))]
    results=[]; all_cids=set(); openmol_xlarge=set(); intersections=[]
    for p,k in jobs:
        r,cids,seen=audit(p,k); results.append(r)
        if k=='mollama': all_cids.update(cids)
        if k=='pubchem': r['cids_shared_with_mollama']=len(cids & all_cids)
        if k=='openmol':
            if p.parent.name=='xlarge': openmol_xlarge=seen
            else: intersections.append({'size':p.parent.name,'unique_pairs':len(seen),'shared_with_xlarge':len(seen&openmol_xlarge)})
        (OUT/'profile.json').write_text(json.dumps({'method':'Exact rows and text-pair hashes; seeded reservoir token estimates. No system/chat-template tokens. PubChem first=original, second=enriched (alternative text views); MolLLaMA first=full dialog, second=assistant-only; other first=prompt, second=answer. Processed diagnostic only: excludes history/molecule input and is not a ready extraction recipe.','results':results,'mollama_unique_cids':len(all_cids),'openmol_overlap':intersections},ensure_ascii=False,indent=2))

if __name__=='__main__': main()

```

## check_chemcot_fallback.py

```python
"""Recover task-dependent CoT fields; choose one view per record, never concatenate duplicates."""
import json,re
from collections import Counter
import audit

def flatten(x):
    if isinstance(x,dict): return '\n'.join(str(k)+': '+flatten(v) for k,v in x.items())
    if isinstance(x,list): return '\n'.join(flatten(v) for v in x)
    return str(x)

def select(r):
    for key in ('raw_cot','struct_cot','cot_result'):
        s=r.get(key)
        if s and str(s).strip():
            s=str(s).strip()
            if key!='raw_cot':
                s=re.sub(r'^```(?:json)?\s*|\s*```$','',s)
                try: s=flatten(json.loads(s))
                except json.JSONDecodeError: pass
            return key,s
    return 'missing',''

audit.texts=lambda r,k:(str(r.get('query') or ''),select(r)[1])
audit.N_SAMPLE=512
result=[]
for p in sorted((audit.ROOT/'Downloads/Train/Chemcot').glob('*/*.json')):
    r,_,_=audit.audit(p,'chemcot')
    r['selected_fields']=dict(Counter(select(x)[0] for x in audit.rows(p)))
    result.append(r)
    (audit.OUT/'chemcot_selected.json').write_text(json.dumps({'method':'raw_cot preferred, otherwise flattened struct_cot, otherwise cot_result stripped of code fences and flattened if JSON parses; no reference/meta/gt injected. Exact counts; 512-record seeded reservoir per file for token estimates.','results':result},ensure_ascii=False,indent=2))

```

## check_overlap.py

```python
"""Conservative exact-match overlap checks; no claim of exhaustive chemical deduplication."""
import hashlib,json,re
from pathlib import Path
from audit import ROOT,OUT,rows

def norm(s): return re.sub(r'\s+',' ',str(s)).strip()
def sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(4*1024*1024),b''):h.update(b)
    return h.hexdigest()

pilot=[json.loads(s) for s in (OUT.parents[2]/'Pilot_v2/experiments/post_token_layer26/prepared/records.jsonl').open()]
questions={norm(r['input_text']):(r['task'],r['question_id'],r['split']) for r in pilot}
caption_ids={r['question_id'] for r in pilot if r['task'] in ('cap2mol','mol2cap')}
result={'method':'Exact whitespace-normalized question matches; CID matches only for Pilot cap2mol/mol2cap, where question IDs are PubChem IDs. No canonical-molecule or semantic near-duplicate checks performed. CID overlap means shared molecules, not necessarily identical questions.','pilot_unique_questions':len(questions),'pilot_caption_cids':len(caption_ids),'copies':[],'sources':[]}
base=ROOT/'Molecule/Baselines/Mol-LLaMA/data'
jobs=[(p,'query') for p in sorted((ROOT/'Downloads/Train/Chemcot').glob('*/*.json'))]
jobs += [(p,'Instruction') for p in sorted((ROOT/'Downloads/Train/OpenMolIns').glob('*/train.csv'))]
jobs += [(p,None) for p in sorted((base/'Mol-LLaMA-Instruct').glob('*.json')) if p.name!='pubchem-molecules.json']
jobs += [(base/'ChEBI-20/train.txt','description')]
for p,field in jobs:
    hits=set(); cid_hits=set(); cue_examples=[]
    for r in rows(p):
        if field and norm(r.get(field,'')) in questions: hits.add(questions[norm(r[field])])
        cid=str(r.get('cid',r.get('CID','')))
        if cid in caption_ids:cid_hits.add(cid)
        if field=='query' and len(cue_examples)<2:
            s=r.get('raw_cot') or ''; m=re.search(r'ground[ -]truth|reference answer|correct answer is',s,re.I)
            if m:cue_examples.append({'id':r.get('id'),'excerpt':s[max(0,m.start()-80):m.end()+160]})
    result['sources'].append({'path':str(p.relative_to(ROOT)),'exact_pilot_question_matches':sorted(hits),'shared_caption_cids':sorted(cid_hits),'answer_cue_examples':cue_examples})
for rel_a,rel_b in [
 ('Downloads/Train/Chemcot/mol_edit/add.json','Construction/Datasets/Chemcot/mol_edit/add.json'),
 ('Downloads/Train/OpenMolIns/xlarge/train.csv','Construction/TOMG-Bench/data/OpenMolIns/xlarge/train.csv'),
 ('Molecule/Baselines/Mol-LLaMA/data/Mol-LLaMA-Instruct/comprehensive_conversations.json','Construction/Datasets/Mol-LLaMA-Instruct/comprehensive_conversations.json')]:
    a,b=ROOT/rel_a,ROOT/rel_b
    if a.exists() and b.exists():
        ha,hb=sha(a),sha(b);result['copies'].append({'a':rel_a,'b':rel_b,'sha256_a':ha,'sha256_b':hb,'identical':ha==hb})
(OUT/'overlap.json').write_text(json.dumps(result,ensure_ascii=False,indent=2))
print(json.dumps({'copy_checks':result['copies'],'sources_with_overlap':[{k:v for k,v in r.items() if k!='answer_cue_examples'} for r in result['sources'] if r['exact_pilot_question_matches'] or r['shared_caption_cids']]},ensure_ascii=False))

```

In [ ]:
# Optional reproducible rerun (CPU only):
import subprocess, sys
audit_dir = '/mnt_nas1/haoqian/Data/Molecule/SAEs/audits/corpus_inventory'
# for script in ("audit.py", "check_chemcot_fallback.py", "check_overlap.py"):
#     subprocess.run([sys.executable, f"{audit_dir}/{script}"], check=True, cwd=audit_dir)


In [1]:
import json
from pathlib import Path
from IPython.display import display, JSON
root = Path(audit_dir)
p = json.loads((root / "profile.json").read_text())
c = json.loads((root / "chemcot_selected.json").read_text())
rows = c["results"] + [r for r in p["results"] if r["kind"] != "chemcot"]
display(JSON([{"kind": r["kind"], "path": r["path"], "rows": r["rows"], "estimated_second_tokens": r["tokens"]["second"]["estimated_total"]} for r in rows]))


[
  {
    "kind": "chemcot",
    "path": "Downloads/Train/Chemcot/mol_edit/add.json",
    "rows": 1499,
    "estimated_second_tokens": 4164245
  },
  {
    "kind": "chemcot",
    "path": "Downloads/Train/Chemcot/mol_edit/delete.json",
    "rows": 1499,
    "estimated_second_tokens": 4122879
  },
  {
    "kind": "chemcot",
    "path": "Downloads/Train/Chemcot/mol_edit/sub.json",
    "rows": 1499,
    "estimated_second_tokens": 4302066
  },
  {
    "kind": "chemcot",
    "path": "Downloads/Train/Chemcot/mol_opt/drd.json",
    "rows": 1500,
    "estimated_second_tokens": 1687881
  },
  {
    "kind": "chemcot",
    "path": "Downloads/Train/Chemcot/mol_opt/gsk.json",
    "rows": 724,
    "estimated_second_tokens": 972278
  },
  {
    "kind": "chemcot",
    "path": "Downloads/Train/Chemcot/mol_opt/jnk.json",
    "rows": 180,
    "estimated_second_tokens": 268592
  },
  {
    "kind": "chemcot",
    "path": "Downloads/Train/Chemcot/mol_opt/logp.json",
    "rows": 1394,
    "estimated_second_to